<a href="https://colab.research.google.com/github/Prasad3617/Python_Practice/blob/main/Supply_Sight_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Here is the complete design and implementation for a production-grade, headless industrial video analytics system. It is structured for reliability, auditability, and local-first execution.

### A. High-Level Architecture Overview

The system is designed as a modular, single-node pipeline running locally (Edge). It operates entirely headless, meaning there is no GUI or visualization output (`cv2.imshow`) that would waste CPU cycles or pose a security risk in a production server environment.

1. **Ingestion**: A background thread continuously captures frames from an RTSP stream or webcam. If the stream lags, older frames are dropped to ensure the pipeline always processes the latest data.
2. **Pre-processing (Motion Gate)**: Before passing frames to the heavy neural network, a lightweight OpenCV background subtractor checks for pixel variance. If no motion is detected, the frame is skipped.
3. **Inference & Tracking**: A local object detector (YOLO) identifies objects. A centroid tracker links these bounding boxes across frames to assign stable IDs.
4. **Business Logic Engine**: Tracks are evaluated against spatial-temporal rules (polygons, lines, time windows).
5. **Event Management**: Valid rule violations trigger events. An event manager deduplicates them using a cooldown system to prevent alert flooding.
6. **Storage & Dispatch**: Events are logged to a JSONL file with associated JPEG snapshot evidence. Alerts are dispatched to system logs (or webhooks).

### B. ASCII Block Diagram

```text
[ RTSP / Camera ]
       |
       v
+-------------------+      +------------------+
| Capture Worker    | ---> | Frame Queue      | (Drops old frames)
| (Threaded, Retry) |      +------------------+
+-------------------+               |
                                    v
                           +------------------+
                           | Motion Gate      | (Skips unchanged frames)
                           +------------------+
                                    |
                                    v
                           +------------------+
                           | YOLO Detector    | (Outputs BBoxes)
                           +------------------+
                                    |
                                    v
                           +------------------+
                           | Centroid Tracker | (Assigns stable IDs)
                           +------------------+
                                    |
                                    v
                           +------------------+
                           | Rule Engine      | (Evaluates zones, lines)
                           +------------------+
                                    |
                                    v
                           +------------------+
                           | Event Manager    | (Cooldown & Deduplication)
                           +------------------+
                                    |
                     +--------------+--------------+
                     |                             |
                     v                             v
            +------------------+          +------------------+
            | Storage Manager  |          | Alert Dispatcher |
            | (Snapshots/JSONL)|          | (Logs, Webhooks) |
            +------------------+          +------------------+

```

### C. Project Tree

```text
industrial_monitor/
├── app.py
├── capture.py
├── motion.py
├── detector.py
├── tracker.py
├── rules.py
├── events.py
├── storage.py
├── alerts.py
├── health.py
├── config.yaml
├── requirements.txt
├── models/
│   └── yolov8n.pt          # Auto-downloaded by Ultralytics or placed manually
└── events/                 # Created automatically for storage

```

### D. requirements.txt

```text
opencv-python-headless>=4.8.0
ultralytics>=8.0.0
PyYAML>=6.0
numpy>=1.24.0

```

### E. config.yaml schema

```yaml
camera:
  source: 0  # Use 0 for local webcam, or "rtsp://admin:pass@ip:port/stream"
  width: 1280
  height: 720
  reconnect_seconds: 3
  queue_size: 2

motion_gate:
  enabled: true
  min_area: 1500
  history: 300
  var_threshold: 32

model:
  path: "models/yolov8n.pt"
  conf_threshold: 0.45
  iou_threshold: 0.50
  classes: ["person", "car", "truck"] # Only process these classes

tracking:
  max_distance: 80
  max_missed_frames: 20

storage:
  event_dir: "events"
  save_snapshot: true
  metadata_file: "events/events.jsonl"

alerts:
  console: true

health:
  log_every_seconds: 60

rules:
  restricted_zones:
    - name: "server_rack_area"
      polygon: [[100, 100], [500, 100], [500, 400], [100, 400]]
      classes: ["person"]
      dwell_seconds: 2
      cooldown_seconds: 30

  line_crossing:
    - name: "loading_dock_door"
      p1: [600, 100]
      p2: [600, 800]
      classes: ["truck"]
      direction: "any" # "any", "left_to_right", "right_to_left"
      cooldown_seconds: 15

  after_hours:
    enabled: true
    start: "18:00"
    end: "06:00"
    classes: ["person"]

```

### F. Full Python Starter Code

**`app.py`**

In [ ]:
import time
import yaml
import logging
from capture import CaptureWorker
from motion import MotionGate
from detector import YOLODetector
from tracker import CentroidTracker
from rules import RuleEngine
from events import EventManager
from storage import StorageManager
from alerts import AlertDispatcher
from health import HealthMonitor

def load_config(path="config.yaml"):
    with open(path, "r", encoding="utf-8") as f:
        return yaml.safe_load(f)

def setup_logging():
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s | %(levelname)s | %(name)s | %(message)s"
    )

def main():
    setup_logging()
    logger = logging.getLogger("main")
    cfg = load_config("config.yaml")

    camera_cfg = cfg.get("camera", {})

    capture = CaptureWorker(
        source=camera_cfg.get("source", 0),
        width=camera_cfg.get("width"),
        height=camera_cfg.get("height"),
        reconnect_seconds=camera_cfg.get("reconnect_seconds", 3),
        queue_size=camera_cfg.get("queue_size", 2),
    )

    motion_gate = MotionGate(cfg.get("motion_gate", {}))
    detector = YOLODetector(cfg.get("model", {}))
    tracker = CentroidTracker(cfg.get("tracking", {}))
    rule_engine = RuleEngine(cfg)
    event_manager = EventManager()
    storage = StorageManager(cfg.get("storage", {}))
    alerts = AlertDispatcher(cfg.get("alerts", {}))
    health = HealthMonitor(cfg.get("health", {}))

    capture.start()
    logger.info("Industrial monitor started in headless mode.")

    try:
        while True:
            frame = capture.read(timeout=2.0)
            if frame is None:
                continue

            health.record_frame()

            if not motion_gate.has_motion(frame):
                continue

            detections = detector.predict(frame)
            tracks = tracker.update(detections)

            candidate_events = rule_engine.evaluate(tracks)
            new_events = event_manager.filter_new(candidate_events)

            for event in new_events:
                snapshot_path = storage.save_event(event, frame)
                alerts.send(event, snapshot_path=snapshot_path)
                health.record_event()

            time.sleep(0.01) # Prevent CPU pegging

    except KeyboardInterrupt:
        logger.info("Shutting down...")
    finally:
        capture.stop()

if __name__ == "__main__":
    main()

**`capture.py`**

In [ ]:
import cv2
import time
import queue
import threading
import logging

class CaptureWorker:
    def __init__(self, source, width=None, height=None, reconnect_seconds=3, queue_size=2):
        # Cast source to int if it's a digit (for local webcam)
        self.source = int(source) if str(source).isdigit() else source
        self.width = width
        self.height = height
        self.reconnect_seconds = reconnect_seconds
        self.frame_queue = queue.Queue(maxsize=queue_size)
        self.stop_event = threading.Event()
        self.thread = None
        self.cap = None
        self.logger = logging.getLogger("capture")

    def start(self):
        self.thread = threading.Thread(target=self._run, daemon=True)
        self.thread.start()

    def stop(self):
        self.stop_event.set()
        if self.thread:
            self.thread.join(timeout=2)
        if self.cap:
            self.cap.release()

    def read(self, timeout=1.0):
        try:
            return self.frame_queue.get(timeout=timeout)
        except queue.Empty:
            return None

    def _open(self):
        self.cap = cv2.VideoCapture(self.source)
        if self.width:
            self.cap.set(cv2.CAP_PROP_FRAME_WIDTH, self.width)
        if self.height:
            self.cap.set(cv2.CAP_PROP_FRAME_HEIGHT, self.height)

        ok = self.cap.isOpened()
        if ok:
            self.logger.info("Connected to camera source: %s", self.source)
        else:
            self.logger.warning("Failed to open camera source: %s", self.source)
        return ok

    def _run(self):
        while not self.stop_event.is_set():
            if self.cap is None or not self.cap.isOpened():
                if not self._open():
                    time.sleep(self.reconnect_seconds)
                    continue

            ok, frame = self.cap.read()
            if not ok or frame is None:
                self.logger.warning("Frame read failed. Reconnecting...")
                self.cap.release()
                self.cap = None
                time.sleep(self.reconnect_seconds)
                continue

            # Drop oldest frame if queue is full to prevent pipeline lag
            if self.frame_queue.full():
                try:
                    self.frame_queue.get_nowait()
                except queue.Empty:
                    pass
            self.frame_queue.put_nowait(frame)

**`motion.py`**

In [ ]:
import cv2

class MotionGate:
    def __init__(self, cfg):
        self.enabled = cfg.get("enabled", True)
        self.min_area = cfg.get("min_area", 1500)
        self.subtractor = cv2.createBackgroundSubtractorMOG2(
            history=cfg.get("history", 300),
            varThreshold=cfg.get("var_threshold", 32),
            detectShadows=False # Disabled for speed unless specifically needed
        )

    def has_motion(self, frame):
        if not self.enabled:
            return True

        mask = self.subtractor.apply(frame)
        _, mask = cv2.threshold(mask, 200, 255, cv2.THRESH_BINARY)

        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        total_area = sum(cv2.contourArea(c) for c in contours)

        return total_area >= self.min_area

**`detector.py`**

In [ ]:
from dataclasses import dataclass
from ultralytics import YOLO

@dataclass
class Detection:
    bbox: tuple   # (x1, y1, x2, y2)
    conf: float
    class_id: int
    class_name: str

class YOLODetector:
    def __init__(self, cfg):
        model_path = cfg.get("path", "yolov8n.pt")
        self.model = YOLO(model_path)
        self.conf_threshold = cfg.get("conf_threshold", 0.45)
        self.iou_threshold = cfg.get("iou_threshold", 0.50)
        self.allowed_classes = set(cfg.get("classes", []))
        self.names = self.model.names

    def predict(self, frame):
        results = self.model.predict(
            source=frame,
            conf=self.conf_threshold,
            iou=self.iou_threshold,
            verbose=False
        )[0]

        detections = []
        if results.boxes is None:
            return detections

        xyxy = results.boxes.xyxy.cpu().numpy()
        confs = results.boxes.conf.cpu().numpy()
        classes = results.boxes.cls.cpu().numpy()

        for box, conf, cls_id in zip(xyxy, confs, classes):
            class_id = int(cls_id)
            class_name = self.names[class_id]

            if self.allowed_classes and class_name not in self.allowed_classes:
                continue

            x1, y1, x2, y2 = map(int, box.tolist())
            detections.append(
                Detection(
                    bbox=(x1, y1, x2, y2),
                    conf=float(conf),
                    class_id=class_id,
                    class_name=class_name
                )
            )

        return detections

**`tracker.py`**

In [ ]:
import math
import time
from dataclasses import dataclass, field

@dataclass
class Track:
    track_id: int
    class_name: str
    bbox: tuple
    centroid: tuple
    first_seen: float
    last_seen: float
    missed_frames: int = 0
    history: list = field(default_factory=list)

class CentroidTracker:
    def __init__(self, cfg):
        self.max_distance = cfg.get("max_distance", 80)
        self.max_missed_frames = cfg.get("max_missed_frames", 20)
        self.next_id = 1
        self.tracks = {}

    @staticmethod
    def _centroid(bbox):
        x1, y1, x2, y2 = bbox
        return ((x1 + x2) // 2, (y1 + y2) // 2)

    @staticmethod
    def _distance(a, b):
        return math.hypot(a[0] - b[0], a[1] - b[1])

    def update(self, detections):
        now = time.time()
        det_centroids = [self._centroid(d.bbox) for d in detections]
        assigned_tracks = set()
        assigned_dets = set()

        for det_idx, det in enumerate(detections):
            best_track_id = None
            best_dist = float("inf")

            for track_id, track in self.tracks.items():
                if track_id in assigned_tracks:
                    continue
                if track.class_name != det.class_name:
                    continue

                dist = self._distance(track.centroid, det_centroids[det_idx])
                if dist < best_dist and dist <= self.max_distance:
                    best_dist = dist
                    best_track_id = track_id

            if best_track_id is not None:
                track = self.tracks[best_track_id]
                track.bbox = det.bbox
                track.centroid = det_centroids[det_idx]
                track.last_seen = now
                track.missed_frames = 0
                track.history.append(track.centroid)
                track.history = track.history[-20:] # Keep last 20 points
                assigned_tracks.add(best_track_id)
                assigned_dets.add(det_idx)

        # New tracks
        for det_idx, det in enumerate(detections):
            if det_idx in assigned_dets:
                continue

            centroid = det_centroids[det_idx]
            self.tracks[self.next_id] = Track(
                track_id=self.next_id,
                class_name=det.class_name,
                bbox=det.bbox,
                centroid=centroid,
                first_seen=now,
                last_seen=now,
                history=[centroid],
            )
            self.next_id += 1

        # Age unmatched tracks
        to_delete = []
        for track_id, track in list(self.tracks.items()):
            if track_id not in assigned_tracks and track.last_seen != now:
                track.missed_frames += 1
                if track.missed_frames > self.max_missed_frames:
                    to_delete.append(track_id)

        for track_id in to_delete:
            del self.tracks[track_id]

        return list(self.tracks.values())

**`rules.py`**

In [ ]:
import cv2
import numpy as np
from datetime import datetime

def inside_polygon(point, polygon):
    poly = np.array(polygon, dtype=np.int32)
    return cv2.pointPolygonTest(poly, point, False) >= 0

def parse_hhmm(value):
    hour, minute = map(int, value.split(":"))
    return hour, minute

def is_within_time_window(now_dt, start_str, end_str):
    sh, sm = parse_hhmm(start_str)
    eh, em = parse_hhmm(end_str)

    start_minutes = sh * 60 + sm
    end_minutes = eh * 60 + em
    now_minutes = now_dt.hour * 60 + now_dt.minute

    if start_minutes <= end_minutes:
        return start_minutes <= now_minutes <= end_minutes
    return now_minutes >= start_minutes or now_minutes <= end_minutes

def side_of_line(point, p1, p2):
    x, y = point
    x1, y1 = p1
    x2, y2 = p2
    return (x - x1) * (y2 - y1) - (y - y1) * (x2 - x1)

class RuleEngine:
    def __init__(self, cfg):
        self.rules_cfg = cfg.get("rules", {})

    def evaluate(self, tracks):
        now = datetime.now()
        events = []

        # 1. Restricted Zones
        for zone in self.rules_cfg.get("restricted_zones", []):
            zone_name = zone["name"]
            polygon = zone["polygon"]
            classes = set(zone.get("classes", []))
            dwell_seconds = zone.get("dwell_seconds", 0)
            cooldown = zone.get("cooldown_seconds", 20)

            for track in tracks:
                if classes and track.class_name not in classes:
                    continue

                if inside_polygon(track.centroid, polygon):
                    age = track.last_seen - track.first_seen
                    if age >= dwell_seconds:
                        events.append({
                            "rule": "restricted_zone",
                            "name": zone_name,
                            "track_id": track.track_id,
                            "class_name": track.class_name,
                            "centroid": track.centroid,
                            "cooldown_seconds": cooldown,
                            "message": f"{track.class_name} in restricted zone '{zone_name}'"
                        })

        # 2. After-hours
        after_hours = self.rules_cfg.get("after_hours", {})
        if after_hours.get("enabled", False):
            if is_within_time_window(now, after_hours["start"], after_hours["end"]):
                classes = set(after_hours.get("classes", []))
                for track in tracks:
                    if classes and track.class_name not in classes:
                        continue
                    events.append({
                        "rule": "after_hours",
                        "name": "after_hours_activity",
                        "track_id": track.track_id,
                        "class_name": track.class_name,
                        "centroid": track.centroid,
                        "cooldown_seconds": 60,
                        "message": f"After-hours {track.class_name} detected"
                    })

        # 3. Line crossing
        for line_rule in self.rules_cfg.get("line_crossing", []):
            name = line_rule["name"]
            p1, p2 = tuple(line_rule["p1"]), tuple(line_rule["p2"])
            classes = set(line_rule.get("classes", []))
            direction = line_rule.get("direction", "any")

            for track in tracks:
                if len(track.history) < 2:
                    continue
                if classes and track.class_name not in classes:
                    continue

                prev_pt = track.history[-2]
                curr_pt = track.history[-1]

                prev_side = side_of_line(prev_pt, p1, p2)
                curr_side = side_of_line(curr_pt, p1, p2)

                crossed = (prev_side < 0 and curr_side > 0) or (prev_side > 0 and curr_side < 0)
                if crossed:
                    dx = curr_pt[0] - prev_pt[0]
                    if direction == "left_to_right" and dx <= 0: continue
                    if direction == "right_to_left" and dx >= 0: continue

                    events.append({
                        "rule": "line_crossing",
                        "name": name,
                        "track_id": track.track_id,
                        "class_name": track.class_name,
                        "centroid": track.centroid,
                        "cooldown_seconds": line_rule.get("cooldown_seconds", 10),
                        "message": f"{track.class_name} crossed line '{name}'"
                    })

        return events

**`events.py`**

In [ ]:
import time
import uuid

class EventManager:
    def __init__(self):
        self.last_sent = {}

    def filter_new(self, candidate_events):
        now = time.time()
        approved = []

        for event in candidate_events:
            # Unique identifier for the violation scenario
            key = (event["rule"], event["name"], event["track_id"])
            cooldown = event.get("cooldown_seconds", 10)
            last_time = self.last_sent.get(key, 0)

            if now - last_time >= cooldown:
                event["event_id"] = str(uuid.uuid4())
                event["timestamp"] = now
                approved.append(event)
                self.last_sent[key] = now

        # Prune memory
        self.last_sent = {k: v for k, v in self.last_sent.items() if now - v < 3600}

        return approved

**`storage.py`**

In [ ]:
import os
import json
import cv2
from datetime import datetime

class StorageManager:
    def __init__(self, cfg):
        self.event_dir = cfg.get("event_dir", "events")
        self.save_snap = cfg.get("save_snapshot", True)
        self.metadata_file = cfg.get("metadata_file", "events/events.jsonl")

        os.makedirs(self.event_dir, exist_ok=True)
        os.makedirs(os.path.dirname(self.metadata_file), exist_ok=True)

    def save_event(self, event, frame):
        snapshot_path = None

        if self.save_snap:
            ts = datetime.now().strftime("%Y%m%d_%H%M%S")
            filename = f"{event['rule']}_{event['name']}_{event['track_id']}_{ts}.jpg"
            snapshot_path = os.path.join(self.event_dir, filename)

            # Optional: Draw tracking info on the snapshot for evidence review
            evidence_frame = frame.copy()
            cx, cy = event["centroid"]
            cv2.circle(evidence_frame, (cx, cy), 5, (0, 0, 255), -1)
            cv2.putText(evidence_frame, event["message"], (10, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

            cv2.imwrite(snapshot_path, evidence_frame)

        record = dict(event)
        record["snapshot_path"] = snapshot_path

        with open(self.metadata_file, "a", encoding="utf-8") as f:
            f.write(json.dumps(record) + "\n")

        return snapshot_path

**`alerts.py`**

In [ ]:
import logging

class AlertDispatcher:
    def __init__(self, cfg):
        self.console = cfg.get("console", True)
        self.logger = logging.getLogger("alerts")

    def send(self, event, snapshot_path=None):
        # In a real system, you would post to a Webhook (Slack, Teams, PagerDuty) here
        if self.console:
            self.logger.warning(
                "ALERT | id=%s | rule=%s:%s | class=%s | track=%s | snapshot=%s | msg=%s",
                event.get("event_id")[:8],
                event.get("rule"),
                event.get("name"),
                event.get("class_name"),
                event.get("track_id"),
                snapshot_path,
                event.get("message")
            )

**`health.py`**

In [ ]:
import time
import logging

class HealthMonitor:
    def __init__(self, cfg):
        self.log_every_seconds = cfg.get("log_every_seconds", 60)
        self.logger = logging.getLogger("health")
        self.start_time = time.time()
        self.last_log_time = time.time()
        self.frames = 0
        self.events = 0

    def record_frame(self):
        self.frames += 1
        self._maybe_log()

    def record_event(self):
        self.events += 1

    def _maybe_log(self):
        now = time.time()
        if now - self.last_log_time >= self.log_every_seconds:
            uptime = now - self.start_time
            fps = self.frames / (now - self.last_log_time)

            self.logger.info("Health | uptime=%.1fs | fps=%.2f | total_events=%d",
                             uptime, fps, self.events)

            self.frames = 0 # Reset frame counter for accurate rolling FPS
            self.last_log_time = now

---

### G. Explanation of how the program works in simple language

This system acts as a tireless virtual security guard.

1. **The Eye**: A background worker grabs video frames from the camera as fast as it can. If the system slows down, the worker drops old frames so it always sees "Right Now".
2. **The Attention Filter**: Before doing heavy thinking, it checks if any pixels have changed (motion detection). If the room is empty and still, it skips the heavy work.
3. **The Brain**: It runs a Machine Learning model (YOLO) to figure out *what* the moving objects are (e.g., a person, a truck) and *where* they are in the frame.
4. **The Memory**: The tracker assigns a unique ID to every object. It remembers that "Person 1" in frame 1 is the same "Person 1" in frame 2.
5. **The Rules**: The rule engine checks the object's path against configured boundaries. Did "Person 1" stay in the restricted zone for more than 2 seconds? Did they cross the loading dock line?
6. **The Logbook**: If a rule is broken, it takes a picture, draws a dot where the object was, and writes a log into a text file. It uses "cooldowns" so it doesn't spam you with 50 logs for the same person lingering in a zone.

### H. Failure modes and mitigations

1. **RTSP Camera Disconnects**: `capture.py` catches frame read failures and enters a reconnect loop, preventing a crash.
2. **Pipeline processing slower than Camera FPS**: Handled by the `queue.Queue(maxsize=2)`. If processing gets bogged down, old frames are silently dropped (`get_nowait`) before putting new ones in.
3. **Alert Flooding**: Handled by the `EventManager`. A single track ID violating a rule triggers an alert *once*, then goes on cooldown.
4. **Out of Disk Space**: *Mitigation needed in production.* A cron job or an added storage rotation policy inside `StorageManager` should delete JPEGs older than X days.
5. **Memory Leaks**: Bounding box dictionaries and tracking histories are aggressively pruned. In `events.py`, the `last_sent` dictionary is pruned of stale cooldowns periodically.

### I. Notes for Windows service / Linux daemon deployment

**Linux (systemd)**:
Create `/etc/systemd/system/industrial-monitor.service`:

```ini
[Unit]
Description=Industrial Video Analytics
After=network.target

[Service]
Type=simple
User=analytics_user
WorkingDirectory=/opt/industrial_monitor
ExecStart=/opt/industrial_monitor/venv/bin/python app.py
Restart=always
RestartSec=5

[Install]
WantedBy=multi-user.target

```

Run `systemctl enable --now industrial-monitor.service`.

**Windows**:
Use **NSSM (Non-Sucking Service Manager)**. It wraps background python scripts reliably.
`nssm install IndustrialMonitor C:\path\to\python.exe C:\path\to\app.py`
Configure the working directory via the NSSM GUI or CLI, and set it to auto-start. No UI will render, which perfectly aligns with our headless architecture.

### J. Suggestions for future upgrades

* **ONNX/OpenVINO**: Swap the PyTorch YOLO model to an exported `yolov8n.onnx` or OpenVINO format. This massively increases FPS on CPU-only edge gateways (like Intel NUCs).
* **SQLite/Postgres**: Migrate from `events.jsonl` to a proper relational database to allow complex querying (e.g., "show me all line crossings by trucks between 2 AM and 4 AM last month").
* **Admin Dashboard**: Attach a lightweight API (like FastAPI) to the DB and serve a React frontend strictly for authorized users to review event logs and configuration.
* **Encryption at Rest**: If the machine is physically accessible on the factory floor, configure OS-level disk encryption (LUKS/BitLocker) or encrypt the JPEG stream before writing to disk to prevent unauthorized viewing of security feeds.
* **Multi-camera Scaling**: Refactor `app.py` into a multi-processing architecture where each camera gets its own dedicated subprocess, feeding structured events back to a centralized `AlertDispatcher` via ZeroMQ or Redis.